# 2 · Find and validate

For one year (built with notebook 1): apply the current rules, rank the papers, and **review them
here**, most likely first: for each paper the evidence, a link to the published version and to an
open copy, and buttons **yes / likely / no / skip**. Every click is saved in your working-data
folder at once: stop whenever you like and continue later.

Then step 7 writes the **corresponding authors** to `contacts.csv` (for an e-mail blast) and gives
a link to file the confirmed papers on GitHub; step 8 learns from your decisions.

| file (in the working-data folder, per year) | what |
|---|---|
| `validate.csv` | the papers to review, with your verdict, reason and note |
| `contacts.csv` | corresponding authors of the confirmed and validated papers: one row per e-mail address |
| `search_misses.csv` | confirmed papers the search did not flag, and why (for whoever maintains the rules) |

**Run the cells one by one, from top to bottom** (▶ at the left of each cell, or Shift + Enter),
and wait for each to finish (a green tick in Colab).

> **Colab users**: nothing to install; step 2 connects your Google Drive, where all results are kept.
>
> **Local users** (JupyterLab): install once in a terminal, from a clone of the repository:
> `pip install -e ".[embed,openiris,notebook]" jupyterlab`, then `jupyter lab notebooks/`.
> Step 1 checks the installation and step 2 does nothing; choose a folder on your computer in step 3.
> (An installed app, e.g. built with LabConstrictor, needs no clone: step 1 downloads the latest
> rules and confirmed papers.)

## 1. Install

As in notebook 1. If Colab shows **Restart session**, click it and run step 1 again.

In [ ]:
# @title 1. Install the code
REPO = 'https://github.com/AIC-Turku/facility-publication-finder'

import os, subprocess, sys
from pathlib import Path

if 'google.colab' in sys.modules:
    print('Google Colab: downloading and installing the code (about a minute) ...')
    CODE = Path('/content/code')          # temporary disk: fresh code, rules and confirmed papers every session
    if CODE.exists():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, str(CODE)], check=True)
    %pip install -q "{CODE}[embed,openiris]"
else:
    try:
        import facility_pubs
    except ImportError:
        raise SystemExit('Not installed: in a terminal, from the repository folder, run\n'
                         '  pip install -e ".[embed,openiris,notebook]" jupyterlab\n'
                         'then restart this notebook.') from None
    CODE = Path(facility_pubs.__file__).resolve().parents[2]

try:
    import ipywidgets  # noqa: F401  (preinstalled in Colab)
except ImportError:
    raise SystemExit('ipywidgets is missing: pip install ipywidgets, then restart the notebook.') from None
from facility_pubs import cli, review, ui, workflow
FACILITIES = ui.facilities_folder(CODE, REPO)   # an installed app downloads the latest rules and papers
print('✅ Code ready:', CODE, '· facilities:', FACILITIES)

## 2. Connect Google Drive

Click **Connect to Google Drive**, choose your work account, allow access and **Continue**.

In [ ]:
# @title 2. Connect your Google Drive (Colab)
if ui.in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive connected: it is under /content/drive/MyDrive')
else:
    print('Running locally: no Google Drive to connect. Choose a folder on this computer in step 3.')

## 3. Settings

Choose the **facility** and the **working-data folder used by notebook 1** (click folders to open
them, **..** to go up; in Colab your Drive is `/content/drive/MyDrive`). The **year** list then shows
the years built in that folder. Usually keep the other values:

* **Check-list papers** (200): how many "reads like facility papers" to include;
* **Papers per page** (1): papers shown at a time in the review;
* **Show again**: decided papers that come back in the review, e.g. *likely* once the authors
  answered (Ctrl/Cmd-click). Usually none.

Then click **Use these settings**.

In [ ]:
# @title 3. Choose the settings
settings = ui.Settings(FACILITIES, kind='review')
settings.widget

## 4. Search the year with the current rules

Prints how many of the year's confirmed papers the rules find (*recall*), and how many new papers
acknowledge the facility (*new ack*) or name one of its instruments (*new instr*).

In [ ]:
# @title 4. Search the year
cfg = settings.require()
cli.main(['rescreen', '--year', str(cfg['year'])])
cli.main(['validate', '--years', str(cfg['year'])])

## 5. Rank

Lists the papers to review, most likely first: those that acknowledge the facility, then those
naming a facility instrument, then the top check-list papers (papers that read like the
facility's papers). The ranking learns from the confirmed papers and from your decisions in other
years (yes as examples, no as counter-examples). Decisions already made are kept.

In [ ]:
# @title 5. Rank
cfg = settings.require()
for path in workflow.prepare(cfg['year'], top_n=cfg['top_n']):
    print('wrote', path)

## 6. Review

For each paper: read why it is listed and the evidence, open a link (new tab), check the
**Methods** and **Acknowledgements**, come back and decide. Optionally choose a **reason** and type
a **note** first: they improve the rules and the ranking.

* **yes**: the work used the facility. It will be filed; its authors go to `contacts.csv`.
* **likely**: probably, not certain. Not filed; its authors go to `contacts.csv` as *likely*, so
  you can ask them (later: *Show again: likely* in step 3).
* **no**: not facility use.
* **skip**: decide later; it comes back next time.

To continue another day: run steps 1, 2, 3 and 6 (4 and 5 are not needed again).

In [ ]:
# @title 6. Review
cfg = settings.require()
review.start(cfg['year'], at_once=cfg['at_once'], revisit=cfg['revisit'])

## 7. Contacts and filing

`contacts.csv` lists the corresponding authors of every confirmed paper of the year and of those
you marked yes or likely, one row per e-mail address. Open it from your Drive folder (or download
it) for the e-mail blast; it is rewritten every time this step runs, so work in a copy. The
addresses were not given by the authors: say where the address comes from, why you write, and how
to opt out; keep a do-not-contact list; delete copies after the campaign.

Then click the **Add papers** link: GitHub opens an issue with the confirmed DOIs filled in.
Submit it and ask a coding agent to file it (e.g. *Assignees → Copilot*); merge its pull request
when the *check papers* check is green.

In [ ]:
# @title 7. Contacts and filing
from IPython.display import display
cfg = settings.require()
contacts, to_file = workflow.collect(cfg['year'], top_n=cfg['top_n'])
print('Corresponding authors:', contacts)
if to_file:
    display(ui.link(workflow.issue_link(REPO, cfg['facility'], to_file),
                    '➡️ Open the Add papers issue (DOIs filled in)'))
    print(to_file)
else:
    print('(no new "yes" verdicts to file)')

## 8. Learn and improve

Re-ranks the other years with your decisions, then summarises them: how often each kind of
candidate was a facility paper, how far down the check list facility papers were still found (to
choose *Check-list papers*), why papers were not facility use, and the papers only the check list
found. Saved as `feedback.txt` in the working-data folder: send it **privately** to whoever
maintains the rules.

In [ ]:
# @title 8. Learn and improve
cfg = settings.require()
for y in workflow.learn(cfg['year'], top_n=cfg['top_n']):
    print('re-ranked', y)
report, path = workflow.feedback()
print(report)
print('\nsaved to', path)